# Probabilistic Risk Analysis

This notebook reproduces figures and tables for probabilistic/synthetic hurricane analysis.

**Paper Figures:**
- **Fig. 3**: Baseline systemic risk (ERA5 10,000-year)
- **Fig. 4**: Climate systemic risk scenarios
- **Fig. 5**: Policy divergence
- **Fig. 6**: Building code sensitivity

**Supplementary Information:**
- **Table 1**: Summary statistics (main text)
- **SI Table 3**: Tail risk metrics (95th percentile)
- **SI Table 4**: Policy sensitivity parameters
- **SI Fig. 1**: Loss return period curves
- **SI Fig. 2**: Climate scenario return period overlay

**Data Requirements:**
- `results/mc_runs/era5_baseline_*/parametric_results_tables.pkl` (Emanuel MC results)
- `results/mc_runs/era5_gcm_*/parametric_results_tables.pkl` (Climate scenario results)

In [ ]:
# Setup
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
from pathlib import Path
from scipy import stats

# Paths
RESULTS_DIR = Path('../results')
MC_DIR = RESULTS_DIR / 'mc_runs'
FIGURES_DIR = RESULTS_DIR / 'figures'
TABLES_DIR = RESULTS_DIR / 'tables'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)

# Style settings
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 11

print("Setup complete")

In [ ]:
# Helper: Load Emanuel MC results
def load_mc_results(run_dir):
    """Load Monte Carlo results from a run directory."""
    pkl_path = run_dir / 'parametric_results_tables.pkl'
    if pkl_path.exists():
        with open(pkl_path, 'rb') as f:
            return pickle.load(f)
    return None

def get_latest_run(prefix):
    """Get the most recent run directory matching prefix."""
    runs = sorted(MC_DIR.glob(f'{prefix}*'), key=lambda p: p.stat().st_mtime, reverse=True)
    return runs[0] if runs else None

In [ ]:
# Load baseline ERA5 results
baseline_dir = get_latest_run('era5_baseline')
if baseline_dir:
    baseline_results = load_mc_results(baseline_dir)
    print(f"Loaded baseline from: {baseline_dir.name}")
else:
    print("No baseline results found - run the Emanuel TC workflow first")
    baseline_results = None

---
## Fig. 3: Baseline Systemic Risk (ERA5)

10,000-year Monte Carlo analysis showing insurer defaults and institutional stress distribution.

In [ ]:
# Extract annual metrics from baseline
if baseline_results:
    annual_totals = baseline_results.get('annual_totals', pd.DataFrame())
    
    # Key metrics
    total_damage = annual_totals['total_damage'].values / 1e9 if 'total_damage' in annual_totals else []
    private_defaults = annual_totals['private_defaults_post_group'].values if 'private_defaults_post_group' in annual_totals else []
    citizens_deficit = annual_totals['citizens_residual'].values / 1e9 if 'citizens_residual' in annual_totals else []
    nfip_borrowing = annual_totals['nfip_treasury_borrowing'].values / 1e9 if 'nfip_treasury_borrowing' in annual_totals else []
    
    print(f"Years simulated: {len(total_damage):,}")
    print(f"Mean annual damage: ${np.mean(total_damage):.1f}B")
    print(f"Max annual damage: ${np.max(total_damage):.1f}B")

In [ ]:
# Fig. 3: Systemic risk baseline
if baseline_results:
    fig, axes = plt.subplots(1, 3, figsize=(12, 4))
    
    # Panel a: Total damage distribution
    ax = axes[0]
    ax.hist(total_damage, bins=50, alpha=0.7, color='steelblue', edgecolor='white')
    ax.axvline(np.percentile(total_damage, 95), color='red', linestyle='--', label='95th percentile')
    ax.axvline(np.percentile(total_damage, 99), color='darkred', linestyle=':', label='99th percentile')
    ax.set_xlabel('Annual Total Damage ($B)')
    ax.set_ylabel('Frequency')
    ax.set_title('a) Total Damage Distribution')
    ax.legend(loc='upper right', fontsize=9)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Panel b: Private insurer defaults
    ax = axes[1]
    default_counts = pd.Series(private_defaults).value_counts().sort_index()
    ax.bar(default_counts.index, default_counts.values, color='#E74C3C', edgecolor='white')
    ax.set_xlabel('Number of Insurer Defaults')
    ax.set_ylabel('Frequency (Years)')
    ax.set_title('b) Insurer Default Distribution')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Panel c: Institutional stress
    ax = axes[2]
    total_institutional = citizens_deficit + nfip_borrowing
    ax.hist(total_institutional[total_institutional > 0], bins=30, alpha=0.7, color='#F39C12', edgecolor='white')
    ax.set_xlabel('Public Institutional Burden ($B)')
    ax.set_ylabel('Frequency')
    ax.set_title('c) Institutional Stress Distribution')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig_era5_systemic_risk_baseline.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {FIGURES_DIR / 'fig_era5_systemic_risk_baseline.png'}")

---
## Fig. 4: Climate Systemic Risk Scenarios

Comparison of systemic risk metrics across GCM-driven climate scenarios.

In [ ]:
# Load climate scenario results
climate_scenarios = ['miroc6', 'mpi', 'cesm2', 'ecearth3']
climate_results = {}

for gcm in climate_scenarios:
    gcm_dir = get_latest_run(f'era5_gcm_{gcm}')
    if gcm_dir:
        results = load_mc_results(gcm_dir)
        if results:
            climate_results[gcm] = results
            print(f"Loaded {gcm}: {gcm_dir.name}")

print(f"\nLoaded {len(climate_results)} climate scenarios")

In [ ]:
# Fig. 4: Climate scenario comparison
if climate_results:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    # Extract metrics for each scenario
    scenarios = ['baseline'] + list(climate_results.keys())
    p95_damage = []
    mean_defaults = []
    
    # Baseline
    if baseline_results:
        annual = baseline_results.get('annual_totals', pd.DataFrame())
        if 'total_damage' in annual:
            p95_damage.append(np.percentile(annual['total_damage'].values / 1e9, 95))
        if 'private_defaults_post_group' in annual:
            mean_defaults.append(annual['private_defaults_post_group'].mean())
    
    # Climate scenarios
    for gcm in climate_results:
        annual = climate_results[gcm].get('annual_totals', pd.DataFrame())
        if 'total_damage' in annual:
            p95_damage.append(np.percentile(annual['total_damage'].values / 1e9, 95))
        if 'private_defaults_post_group' in annual:
            mean_defaults.append(annual['private_defaults_post_group'].mean())
    
    x_pos = np.arange(len(scenarios))
    
    # Panel a: 95th percentile damage
    ax = axes[0]
    colors = ['steelblue'] + ['#E74C3C'] * len(climate_results)
    ax.bar(x_pos, p95_damage, color=colors, edgecolor='white')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([s.upper() for s in scenarios], rotation=45, ha='right')
    ax.set_ylabel('95th Percentile Damage ($B)')
    ax.set_title('a) Tail Risk by Climate Scenario')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Panel b: Mean defaults
    ax = axes[1] 
    ax.bar(x_pos, mean_defaults, color=colors, edgecolor='white')
    ax.set_xticks(x_pos)
    ax.set_xticklabels([s.upper() for s in scenarios], rotation=45, ha='right')
    ax.set_ylabel('Mean Annual Insurer Defaults')
    ax.set_title('b) Systemic Default Risk')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig_climate_systemic_risk_scenarios.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {FIGURES_DIR / 'fig_climate_systemic_risk_scenarios.png'}")

---
## Fig. 5: Policy Divergence

How different policy configurations affect coverage gaps and institutional burden.

In [ ]:
# Load policy sensitivity runs
policy_configs = {
    'current': 'era5_baseline',
    'high_penetration': 'era5_policy_p80',
    'low_penetration': 'era5_policy_p40',
    'expanded_citizens': 'era5_policy_citizens_expanded'
}

policy_results = {}
for policy_name, prefix in policy_configs.items():
    run_dir = get_latest_run(prefix)
    if run_dir:
        results = load_mc_results(run_dir)
        if results:
            policy_results[policy_name] = results
            print(f"Loaded {policy_name}: {run_dir.name}")

print(f"\nLoaded {len(policy_results)} policy configurations")

In [ ]:
# Fig. 5: Policy divergence analysis
if len(policy_results) > 1:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    policies = list(policy_results.keys())
    uninsured_share = []
    public_burden = []
    
    for policy in policies:
        annual = policy_results[policy].get('annual_totals', pd.DataFrame())
        # Calculate uninsured share
        total = annual['total_damage'].sum() if 'total_damage' in annual else 1
        uninsured = annual['uninsured_loss'].sum() if 'uninsured_loss' in annual else 0
        uninsured_share.append(100 * uninsured / total if total > 0 else 0)
        
        # Calculate public burden
        citizens = annual['citizens_residual'].mean() / 1e9 if 'citizens_residual' in annual else 0
        nfip = annual['nfip_treasury_borrowing'].mean() / 1e9 if 'nfip_treasury_borrowing' in annual else 0
        public_burden.append(citizens + nfip)
    
    x_pos = np.arange(len(policies))
    policy_labels = ['Current', 'High Penetration', 'Low Penetration', 'Expanded Citizens']
    colors = ['steelblue', '#2ECC71', '#E74C3C', '#F39C12']
    
    # Panel a: Uninsured share
    ax = axes[0]
    ax.bar(x_pos, uninsured_share, color=colors[:len(policies)], edgecolor='white')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(policy_labels[:len(policies)], rotation=45, ha='right')
    ax.set_ylabel('Uninsured Loss Share (%)')
    ax.set_title('a) Coverage Gap')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Panel b: Public burden
    ax = axes[1]
    ax.bar(x_pos, public_burden, color=colors[:len(policies)], edgecolor='white')
    ax.set_xticks(x_pos)
    ax.set_xticklabels(policy_labels[:len(policies)], rotation=45, ha='right')
    ax.set_ylabel('Mean Public Burden ($B/year)')
    ax.set_title('b) Institutional Exposure')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig_era5_policy_divergence.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {FIGURES_DIR / 'fig_era5_policy_divergence.png'}")
else:
    print("Insufficient policy runs for divergence analysis")

---
## Fig. 6: Building Code Sensitivity

Impact of building code improvements on wind and flood losses.

In [ ]:
# Load building code sensitivity results
buildingcode_dir = get_latest_run('era5_buildingcode')
if buildingcode_dir:
    bc_results = load_mc_results(buildingcode_dir)
    print(f"Loaded building code results: {buildingcode_dir.name}")
else:
    bc_results = None
    print("No building code results found")

In [ ]:
# Fig. 6: Building code sensitivity
if bc_results and baseline_results:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))
    
    # Get comparison data
    baseline_annual = baseline_results.get('annual_totals', pd.DataFrame())
    bc_annual = bc_results.get('annual_totals', pd.DataFrame())
    
    # Panel a: Damage reduction distribution
    ax = axes[0]
    if 'total_damage' in baseline_annual and 'total_damage' in bc_annual:
        baseline_dmg = baseline_annual['total_damage'].values / 1e9
        bc_dmg = bc_annual['total_damage'].values / 1e9
        reduction = 100 * (1 - bc_dmg / baseline_dmg)
        reduction = reduction[np.isfinite(reduction)]
        
        ax.hist(reduction, bins=30, alpha=0.7, color='#2ECC71', edgecolor='white')
        ax.axvline(np.mean(reduction), color='darkgreen', linestyle='--', 
                   label=f'Mean: {np.mean(reduction):.1f}%')
        ax.set_xlabel('Damage Reduction (%)')
        ax.set_ylabel('Frequency')
        ax.set_title('a) Building Code Effectiveness')
        ax.legend()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    # Panel b: Wind vs Flood breakdown
    ax = axes[1]
    categories = ['Wind Loss', 'Flood Loss']
    baseline_vals = []
    bc_vals = []
    
    for col, cat in [('wind_loss', 'Wind Loss'), ('flood_loss', 'Flood Loss')]:
        if col in baseline_annual:
            baseline_vals.append(baseline_annual[col].mean() / 1e9)
        if col in bc_annual:
            bc_vals.append(bc_annual[col].mean() / 1e9)
    
    if baseline_vals and bc_vals:
        x = np.arange(len(categories))
        width = 0.35
        ax.bar(x - width/2, baseline_vals, width, label='Baseline', color='steelblue')
        ax.bar(x + width/2, bc_vals, width, label='Improved Codes', color='#2ECC71')
        ax.set_xticks(x)
        ax.set_xticklabels(categories)
        ax.set_ylabel('Mean Annual Loss ($B)')
        ax.set_title('b) Loss by Peril Type')
        ax.legend()
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig_climate_buildingcode_sensitivity.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {FIGURES_DIR / 'fig_climate_buildingcode_sensitivity.png'}")

---
## Table 1: Summary Statistics

Key metrics from the baseline ERA5 10,000-year simulation.

In [ ]:
# Table 1: Summary statistics
if baseline_results:
    annual = baseline_results.get('annual_totals', pd.DataFrame())
    
    summary_stats = {
        'Metric': [
            'Simulation Years',
            'Mean Annual Total Damage ($B)',
            '95th Percentile Damage ($B)',
            '99th Percentile Damage ($B)',
            'Maximum Annual Damage ($B)',
            'Mean Insurer Defaults/Year',
            'Max Insurer Defaults in Year',
            'Years with Any Default (%)',
            'Mean Citizens Deficit ($B/year)',
            'Mean NFIP Borrowing ($B/year)'
        ],
        'Value': [
            f"{len(annual):,}",
            f"${annual['total_damage'].mean() / 1e9:.1f}",
            f"${np.percentile(annual['total_damage'].values / 1e9, 95):.1f}",
            f"${np.percentile(annual['total_damage'].values / 1e9, 99):.1f}",
            f"${annual['total_damage'].max() / 1e9:.1f}",
            f"{annual['private_defaults_post_group'].mean():.2f}",
            f"{annual['private_defaults_post_group'].max():.0f}",
            f"{100 * (annual['private_defaults_post_group'] > 0).mean():.1f}%",
            f"${annual['citizens_residual'].mean() / 1e9:.2f}",
            f"${annual['nfip_treasury_borrowing'].mean() / 1e9:.2f}"
        ]
    }
    
    df_summary = pd.DataFrame(summary_stats)
    df_summary.to_csv(TABLES_DIR / 'table1_summary_statistics.csv', index=False)
    print("Table 1: Summary Statistics")
    print(df_summary.to_string(index=False))

---
## SI Fig. 1: Loss Return Period Curves

Exceedance probability curves for total damage.

In [ ]:
# SI Fig. 1: Return period curves
if baseline_results:
    annual = baseline_results.get('annual_totals', pd.DataFrame())
    
    if 'total_damage' in annual:
        damages = np.sort(annual['total_damage'].values / 1e9)[::-1]
        n = len(damages)
        return_periods = (n + 1) / np.arange(1, n + 1)
        
        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(return_periods, damages, 'b-', linewidth=1.5, label='Total Damage')
        
        # Mark key return periods
        for rp, color in [(100, 'orange'), (250, 'red'), (500, 'darkred')]:
            idx = np.argmin(np.abs(return_periods - rp))
            ax.axvline(rp, color=color, linestyle='--', alpha=0.5)
            ax.annotate(f'{rp}yr: ${damages[idx]:.0f}B', 
                       xy=(rp, damages[idx]), xytext=(rp*1.2, damages[idx]*1.1),
                       fontsize=9, color=color)
        
        ax.set_xscale('log')
        ax.set_xlabel('Return Period (years)')
        ax.set_ylabel('Annual Total Damage ($B)')
        ax.set_title('Loss Return Period Curve (ERA5 Baseline)')
        ax.grid(True, alpha=0.3)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        
        plt.tight_layout()
        plt.savefig(FIGURES_DIR / 'fig_loss_return_period.png', dpi=300, bbox_inches='tight')
        plt.show()
        print(f"Saved: {FIGURES_DIR / 'fig_loss_return_period.png'}")

---
## SI Fig. 2: Climate Scenario Return Period Overlay

Comparison of return period curves across climate scenarios.

In [ ]:
# SI Fig. 2: Climate return period overlay
if baseline_results and climate_results:
    fig, ax = plt.subplots(figsize=(8, 5))
    
    # Plot baseline
    annual = baseline_results.get('annual_totals', pd.DataFrame())
    if 'total_damage' in annual:
        damages = np.sort(annual['total_damage'].values / 1e9)[::-1]
        n = len(damages)
        return_periods = (n + 1) / np.arange(1, n + 1)
        ax.plot(return_periods, damages, 'b-', linewidth=2, label='ERA5 Baseline')
    
    # Plot climate scenarios
    colors = ['#E74C3C', '#F39C12', '#9B59B6', '#1ABC9C']
    for i, (gcm, results) in enumerate(climate_results.items()):
        annual = results.get('annual_totals', pd.DataFrame())
        if 'total_damage' in annual:
            damages = np.sort(annual['total_damage'].values / 1e9)[::-1]
            n = len(damages)
            return_periods = (n + 1) / np.arange(1, n + 1)
            ax.plot(return_periods, damages, color=colors[i % len(colors)], 
                   linewidth=1.5, label=gcm.upper())
    
    ax.set_xscale('log')
    ax.set_xlabel('Return Period (years)')
    ax.set_ylabel('Annual Total Damage ($B)')
    ax.set_title('Return Period Curves: Climate Scenarios')
    ax.legend(loc='upper left')
    ax.grid(True, alpha=0.3)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / 'fig_return_period_climate_overlay.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"Saved: {FIGURES_DIR / 'fig_return_period_climate_overlay.png'}")

---
## SI Table 3: Tail Risk Metrics (95th Percentile)

Comparison of 95th percentile metrics across scenarios.

In [ ]:
# SI Table 3: Tail risk metrics
all_scenarios = {'Baseline': baseline_results}
all_scenarios.update(climate_results)

tail_data = []
for name, results in all_scenarios.items():
    if results:
        annual = results.get('annual_totals', pd.DataFrame())
        if 'total_damage' in annual:
            tail_data.append({
                'Scenario': name.upper(),
                'P95 Total Damage ($B)': np.percentile(annual['total_damage'].values / 1e9, 95),
                'P95 Insurer Defaults': np.percentile(annual['private_defaults_post_group'].values, 95),
                'P95 Citizens Deficit ($B)': np.percentile(annual['citizens_residual'].values / 1e9, 95),
                'P95 NFIP Borrowing ($B)': np.percentile(annual['nfip_treasury_borrowing'].values / 1e9, 95)
            })

if tail_data:
    df_tail = pd.DataFrame(tail_data)
    df_tail.to_csv(TABLES_DIR / 'si_table3_tail_risk.csv', index=False)
    print("SI Table 3: Tail Risk Metrics (95th Percentile)")
    print(df_tail.round(2).to_string(index=False))

---
## SI Table 4: Policy Sensitivity Parameters

Configuration parameters used in policy sensitivity analysis.

In [ ]:
# SI Table 4: Policy parameters (static reference)
policy_params = pd.DataFrame({
    'Parameter': [
        'Wind Insurance Penetration',
        'Flood Insurance Penetration (SFHA)',
        'Citizens Premium Cap',
        'NFIP Maximum Coverage',
        'FHCF Retention',
        'Private Reinsurance Attachment'
    ],
    'Baseline': ['65%', '40%', '$700k', '$250k', '$8.7B', 'Company-specific'],
    'High Penetration': ['80%', '60%', '$700k', '$250k', '$8.7B', 'Company-specific'],
    'Low Penetration': ['40%', '20%', '$700k', '$250k', '$8.7B', 'Company-specific'],
    'Expanded Citizens': ['65%', '40%', '$1M', '$250k', '$8.7B', 'Company-specific']
})

policy_params.to_csv(TABLES_DIR / 'si_table4_policy_parameters.csv', index=False)
print("SI Table 4: Policy Sensitivity Parameters")
print(policy_params.to_string(index=False))